## <font color="greenblue">Processamento - uma única tabela</font>

In [ ]:
import pandas as pd
import numpy as np
# from statsmodels.stats.proportion import proportions_ztest
from collections import Counter
from utils import ordenar_labels, ordenar_labels_multipla, ordenar_valores, classificar_nps, funcao_agrupamento, classificar_satis

TipoTabela = 'MULTIPLA'
Colunas = 'MERC_PME, MERC_500, TIM_TOT, TIM_PME, TIM_500, TIM_PRIME, VIV_TOT, VIV_PME, VIV_500, CLA_TOT, CLA_PME, CLA_500'
Cabecalho = 'Mercado PME, Mercado 500+, TIM, TIM PME, TIM 500+, TIM Prime, VIVO, VIVO PME, VIVO 500+, CLARO, CLARO PME, CLARO 500+'
Var_linha = 'Q8' 
NS_NR = 'NAO'
valores_BTB = '1, 2'
valores_medium = '3'
valores_TTB = '4, 5'
Valores_Agrup = 'Q8_1, Q8_2, Q8_3'
Fecha_100 = 'NAO'
Var_ID = 'codigo_entrevistado' # codigo_entrevistado | ID_EMP
Var_Pond = 'POND'
Titulo = '8. (SATISFAÇÃO 2021) Por que deu esta nota de recomendação para a _______? (ESPONTÂNEA - RM)' # 8. (SATISFAÇÃO 2021) Por que deu esta nota de recomendação para a _______? (ESPONTÂNEA - RM)

In [8]:
import pandas as pd
import numpy as np
# from statsmodels.stats.proportion import proportions_ztest
from collections import Counter
from utils import ordenar_labels, ordenar_labels_multipla, ordenar_valores, classificar_nps, funcao_agrupamento, classificar_satis

TipoTabela = 'NPS'
Colunas = 'ONDA, EMP_G, VAR_G, GC_G'  # 'ONDA, SEG_NOVO_BU'
Cabecalho = 'Ondas, Empreendedores (PF+PJ), Varejo (PF+PJ), Grandes Contas (PF+PJ)' # 'ONDA, SEGMENTO'
Var_linha = 'NPS_IT' 
NS_NR = 'NAO'
valores_BTB = '1, 2, 3'
valores_TTB = '8, 9, 10'
Valores_Agrup = 'Q8_1, Q8_2, Q8_3'
Fecha_100 = 'NAO'
Var_ID = 'ID_EMP' # codigo_entrevistado | ID_EMP
Var_Pond = 'POND'
Titulo = 'Q4_IT. O quanto você recomendaria o ITAU' # 8. (SATISFAÇÃO 2021) Por que deu esta nota de recomendação para a _______? (ESPONTÂNEA - RM)

In [9]:
path = r'C:\Users\rayner.santos\Downloads\Consistencia e Processamento\BD_TIM_Imagem_Onda1_2025_v10.xlsx'
path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\Cielo\EMP_Cielo Satisfacao_3onda_NOV25_2025.12.26.xlsx"
path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\Cielo\Base de dados atualizada - 2026-03-13 17_03_44.xlsx"
data = pd.read_excel(path, sheet_name='BD_CODIGOS')
df = data.copy()
df

,ID_EMP,ID_ONDA,ONDA,COLETA,NU_SO_EC,POND,CIELO_CAMPO,PF_PJ,VCID,VUF,...,FATC,VRAMO,SETOR_BIGDATA,TEMP_EX,TEMP_AC,SEXO,SEG_ONDA,EMP_G,VAR_G,GC_G
0,3000230,1634,59,2,2.901174e+09,0.999119,2,2,NaN,2,...,2.0,20,20.0,1,1,1,8,NaN,4.0,NaN
1,3001659,5090,59,2,NaN,1.497418,4,2,NaN,7,...,2.0,13,13.0,4,4,1,8,NaN,4.0,NaN
2,3000383,1986,59,2,NaN,0.317763,2,2,NaN,6,...,1.0,21,21.0,2,2,2,4,4.0,NaN,NaN
3,3000455,2153,59,2,NaN,0.263658,4,2,NaN,6,...,1.0,21,21.0,6,5,1,4,4.0,NaN,NaN
4,3000985,3488,59,2,NaN,1.982729,4,1,NaN,2,...,2.0,8,8.0,5,5,1,8,NaN,4.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15381,3004081,102279,59,1,NaN,0.263658,3,2,80.0,8,...,1.0,21,21.0,4,3,1,4,4.0,NaN,NaN
15382,3004082,102294,59,1,NaN,1.982729,3,1,80.0,6,...,2.0,13,13.0,6,3,1,8,NaN,4.0,NaN
15383,3004083,102343,59,1,NaN,1.497418,3,2,4.0,1,...,2.0,4,4.0,5,5,1,8,NaN,4.0,NaN
15384,3004084,102609,59,1,NaN,1.497418,3,2,3.0,5,...,2.0,20,20.0,2,2,1,8,NaN,4.0,NaN


In [10]:
lista_labels = pd.read_excel(path, sheet_name='LISTA_LABELS')
lista_labels = lista_labels.iloc[1:, :].copy()
lista_labels.columns = ['Coluna', 'Codigo', 'Label']
lista_labels["Coluna"] = lista_labels["Coluna"].ffill().str.strip()
lista_labels

,Coluna,Codigo,Label
1,ONDA,28,Abr23
2,ONDA,29,Mai23 (mensal)
3,ONDA,30,Jun23 (mensal)
4,ONDA,31,Jul23
5,ONDA,32,Ago23 (mensal)
...,...,...,...
10245,VAR_G,4,Varejo - Nov25
10246,GC_G,1,GC - Nov24
10247,GC_G,2,GC - Mar25
10248,GC_G,3,GC - Jul25


In [11]:
# Variáveis para as colunas da tabela (bandeiras)
Colunas = Colunas.split(sep=', ')
print(Colunas)
dict_ord_labels = {}

for col in Colunas:
    if col not in df.columns:
        raise ValueError(f"Coluna '{col}' não encontrada no DataFrame.")        
    else:
        df[col], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=col)
        dict_ord_labels[col] = ord_labels
        print(f"\nColuna ordenada (unique): {df[col].unique()}")

['ONDA', 'EMP_G', 'VAR_G', 'GC_G']

#== VARIÁVEL SENDO PROCESSADA: ONDA ==#
ONDA
59    4074
55    3877
51    3826
47    3609
Name: count, dtype: int64 

ONDA
59    0.264786
55    0.251982
51    0.248668
47    0.234564
Name: proportion, dtype: float64

Labels filtrados para a coluna alvo:
     Codigo           Label
1       28           Abr23
2       29  Mai23 (mensal)
3       30  Jun23 (mensal)
4       31           Jul23
5       32  Ago23 (mensal)
6       33  Set23 (mensal)
7       34  Out23 (mensal)
8       35           Nov23
9       36          Dez/23
10      37  Jan24 (mensal)
11      38  Fev24 (mensal)
12      39          Mar/24
13      40  Abr24 (mensal)
14      41  Mai24 (mensal)
15      42  Jun24 (mensal)
16      43          Jul/24
17      44  Ago24 (mensal)
18      45  Set24 (mensal)
19      46  Out24 (mensal)
20      47           Nov24
21      48   Dez24(Mensal)
22      49   Jan25(Mensal)
23      50   Fev25(Mensal)
24      51           Mar25
25      52  Abr25 (Mensal)
26      

In [12]:
# Etapas de ETL para as colunas principais a serem utilizadas
# Transformação na variável para a linha da tabela
if TipoTabela == 'SIMPLES':
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
    else:
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
    
elif (TipoTabela == 'NPS') | (TipoTabela == 'IPA_10'):
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha] = df[Var_linha].replace('ns/nr', np.nan)
        df[Var_linha] = df[Var_linha].replace('90', np.nan)
        df[Var_linha] = df[Var_linha].replace('99', np.nan)
        df[Var_linha] = df[Var_linha].replace('999', np.nan)
        df[Var_linha] = df[Var_linha].replace('9999', np.nan)
        df[Var_linha] = df[Var_linha].replace(90, np.nan)
        df[Var_linha] = df[Var_linha].replace(99, np.nan)
        df[Var_linha] = df[Var_linha].replace(999, np.nan)
        df[Var_linha] = df[Var_linha].replace(9999, np.nan)
        df[Var_linha] = pd.to_numeric(df[Var_linha], errors='coerce', downcast='integer')
        # df[Var_linha], ord_labels = ordenar_labels(df=df, lista_labels=lista_labels, Variavel=Var_linha)
        df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

        if TipoTabela == 'NPS':
            df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

        elif TipoTabela == 'IPA_10':
            valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

    else:
        # df[Var_linha] = df[Var_linha].replace('90', np.nan)
        # df[Var_linha] = df[Var_linha].replace('99', np.nan)
        # df[Var_linha] = df[Var_linha].replace('999', np.nan)
        # df[Var_linha] = df[Var_linha].replace('9999', np.nan)
        # df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

        if TipoTabela == 'NPS':
            df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

        elif TipoTabela == 'IPA_10':
            valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
            valores_medium = [int(v) for v in valores_medium.split(sep=', ')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_medium, valores_TTB)

elif TipoTabela == 'IPA_5':
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha] = df[Var_linha].replace('90', np.nan)
        df[Var_linha] = df[Var_linha].replace('99', np.nan)
        df[Var_linha] = df[Var_linha].replace('999', np.nan)
        df[Var_linha] = df[Var_linha].replace(90, np.nan)
        df[Var_linha] = df[Var_linha].replace(99, np.nan)
        df[Var_linha] = df[Var_linha].replace(999, np.nan)
        valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
        valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
        df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
        df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)
    else:
        valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
        valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
        df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
        df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)

elif TipoTabela == 'MULTIPLA':
    if NS_NR == 'NAO':
        df_NS_NR = df.copy()
        
        Valores_Agrup = Valores_Agrup.split(sep=', ')
        # Converte as colunas de motivos para string, preservando NaN
        # for c in Valores_Agrup:
        #     df[c] = df[c].astype("object").where(df[c].isna(), df[c].astype(str))
        #     df_NS_NR[c] = df_NS_NR[c].astype("object").where(df_NS_NR[c].isna(), df_NS_NR[c].astype(str))

        bd_motivo = pd.melt(df, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('90', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('99', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('999', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('9999', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(90, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(99, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(999, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(9999, np.nan)
        bd_motivo = bd_motivo.dropna(subset=[Var_linha])
        print(f'\nbd_motivo em formato de código:\n{bd_motivo}')
        bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('NS/NR', np.nan)
        # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)  

        df_limpo = bd_motivo.dropna(subset=[Var_linha])
        print(f'bd_motivo finalizado:\n{df_limpo}')
        df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')
                
        # Bancos para realizar o cálculo do Índice de Multiplicidade (incluir NS/NR)
        bd_motivo_NS_NR = pd.melt(df_NS_NR, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo_NS_NR = bd_motivo_NS_NR.dropna(subset=[Var_linha])
        bd_motivo_NS_NR = ordenar_labels_multipla(df=bd_motivo_NS_NR, lista_labels=lista_labels, Variavel=Var_linha, 
                                                  Var_Valores_Agrup=Valores_Agrup[0])
        # bd_motivo_NS_NR[Var_linha] = pd.Categorical(bd_motivo_NS_NR[Var_linha], 
        #                                     categories=ordenar_valores(bd_motivo_NS_NR[Var_linha]), ordered=True)
        
        df_NS_NR_limpo = bd_motivo_NS_NR.dropna(subset=[Var_linha])
        print(f'bd_motivo_NS_NR finalizado:\n{df_limpo}')
        df_NS_NR_unico = df_NS_NR_limpo.drop_duplicates(subset=Var_ID, keep='first')    
        
    else:
        Valores_Agrup = Valores_Agrup.split(sep=',')
        bd_motivo = pd.melt(df, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo = bd_motivo.dropna(subset=[Var_linha])
        bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
        # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], 
        #                                     categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)
        
        df_limpo = bd_motivo.dropna(subset=[Var_linha])
        df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')

In [36]:
label_retirada = "NS/NR"
labels_var_linha = (
    lista_labels.loc[(lista_labels["Coluna"] == Var_linha) & (lista_labels["Label"] != label_retirada), "Codigo"]
    .dropna()
    .tolist()
)
labels_var_linha

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [37]:
label_retirada = "NS/NR"
labels_var_linha = (
    lista_labels.loc[(lista_labels["Coluna"] == Var_linha) & (lista_labels["Label"] != label_retirada), "Codigo"]
    .dropna()
    .tolist()
)
tabela = pd.crosstab(df[Var_linha], df[Colunas[0]], values=df[Var_Pond], aggfunc='sum').fillna(0)
print(len(tabela))
if len(tabela) == 0:
    tabela = pd.DataFrame(0, 
                          index=df[Var_linha][pd.notna(df[Var_linha])].unique(), 
                          columns=df[col][pd.notna(df[col])].unique()
                        )
# garante todas as labels no índice
tabela = tabela.reindex(labels_var_linha, fill_value=0)
print(tabela)


10
ONDA    Nov24  Mar25      Jul25       Nov25
NPS_IT                                     
0         0.0    0.0   0.000000    0.527316
1         0.0    0.0   0.000000    0.000000
2         0.0    0.0   2.525763    0.000000
3         0.0    0.0   1.561476    0.000000
4         0.0    0.0   3.119698    0.000000
5         0.0    0.0   4.135050    8.014406
6         0.0    0.0   3.413134    8.023458
7         0.0    0.0  12.199158   23.178994
8         0.0    0.0  20.035851   61.082203
9         0.0    0.0  23.713579   53.863437
10        0.0    0.0  80.820092  158.968397


In [20]:
#======= Criação da Tabela Geral =======#
tabela_geral = []
tabelas_pond = []
tabelas_pond_freq_abs = []
tabelas_sem_pond = []
aux_tabelas_pond = []
aux_tabelas_sem_pond = []

tbl_pond_freq_abs_respondentes = []
tbl_sem_pond_respondentes = []

tbl_pond_freq_abs_respostas_NS_NR = []
tbl_pond_freq_abs_respondentes_NS_NR = []

if TipoTabela == 'MULTIPLA':
    if NS_NR == 'NAO':
        banco = df_NS_NR_unico
    else:
        banco = df_unico

    if NS_NR == 'NAO':
        banco_pivotado = bd_motivo_NS_NR
    else:
        banco_pivotado = bd_motivo

    i = 0    
    for col in Colunas:

        # Gerar Tabelas Ponderadas de Frequência Absoluta com o banco empilhado
        tabela = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        if Fecha_100 == 'SIM':
            tabela = tabela.div(tabela.sum())
            tabelas_pond.append(tabela)
        else:
            tabelas_pond.append(tabela)

        tabela = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respostas_NS_NR.append(tabela)

        tabela = pd.pivot_table(df_unico, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respondentes.append(tabela)

        tabela = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respondentes_NS_NR.append(tabela)

        tabela = pd.crosstab(df_unico[Var_linha], df_unico[col], dropna=False)
        tabela = tabela.fillna(0)
        if len(tabela) == 0:
            tabela = pd.DataFrame(0, index=df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique(), 
                                columns=df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique())
            print(f'{df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique()}\n')
            print(f'{df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique()}\n')
        print(f'tabela:\n{tabela}')
        tbl_sem_pond_respondentes.append(tabela)
        i += 1

    tabela_geral = pd.concat(tabelas_pond, axis=1)
    tbl_pond_freq_abs_respondentes = pd.concat(tbl_pond_freq_abs_respondentes, axis=1)
    tbl_sem_pond_respondentes = pd.concat(tbl_sem_pond_respondentes, axis=1)
    tbl_pond_freq_abs_respostas_NS_NR = pd.concat(tbl_pond_freq_abs_respostas_NS_NR, axis=1)
    tbl_pond_freq_abs_respondentes_NS_NR = pd.concat(tbl_pond_freq_abs_respondentes_NS_NR, axis=1)
    
    if Fecha_100 != 'SIM':
        soma_base_ponderada = tbl_pond_freq_abs_respondentes.sum()
        tabela_geral = tabela_geral.div(soma_base_ponderada)
    
    print(f'{tabela_geral}\n')


    # Trazendo a coluna de valores gerais
    soma_colunas = tbl_pond_freq_abs_respondentes.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    valores_gerais = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, aggfunc='sum')
    if Fecha_100 == 'SIM':
        percentual_geral = valores_gerais.div(valores_gerais.sum()).sort_index()
        print(f'\npercentual_geral:\n{percentual_geral}')
        print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')
    else:
        percentual_geral = valores_gerais.div(base_ponderada[0]).sort_index()
        print(f'\npercentual_geral:\n{percentual_geral}')
        print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')

    # tabela_geral = tabela_geral.div(base_ponderada[1:])
    tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
    print(f'{tabela_geral}\n')

    # Valores para Base Ponderada
    soma_colunas = tbl_pond_freq_abs_respondentes.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    tabela_geral.loc['Base Ponderada'] = base_ponderada

    # Valores para Base Sem Ponderar
    valores_gerais_respondentes = df_unico[Var_ID].value_counts()
    soma_colunas = pd.Series(tbl_sem_pond_respondentes.sum())
    print(f'\n{soma_colunas}\n')
    base_sem_ponderar = pd.Series(valores_gerais_respondentes.sum())
    base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
    print(f'base sem ponderar:\n{base_sem_ponderar.index}\n')
    print(f'{len(base_sem_ponderar.index)}\n')
    print(f'tabela geral:\n{tabela_geral.columns}\n')
    print(f'{len(tabela_geral.columns)}\n')
    base_sem_ponderar.index = tabela_geral.columns
    tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

    # Valores para Base Ponderada - Respostas
    soma_colunas_respostas = tbl_pond_freq_abs_respostas_NS_NR.sum()
    soma_colunas_respostas = pd.Series(soma_colunas_respostas)
    base_ponderada_respostas = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, aggfunc='sum')
    base_ponderada_respostas = pd.Series(base_ponderada_respostas.sum())
    base_ponderada_respostas = pd.concat([base_ponderada_respostas, soma_colunas_respostas])

    # Valores para Base Ponderada - Respondentes
    soma_colunas = tbl_pond_freq_abs_respondentes_NS_NR.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])

    # Índice de Multiplicidade (Total de respostas / Total de respondentes)
    indice_multiplicidade = base_ponderada_respostas / base_ponderada
    tabela_geral.loc['Índice de Multiplicidade'] = indice_multiplicidade

    tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
    print(f'{tabela_geral}\n')

else:
    if df[Var_linha].isna().all():
        # df[Var_linha] = df[Var_linha].astype('float').fillna(0)
        df[Var_linha] = df[Var_linha].cat.add_categories(['Não possui categoria']).fillna('Não possui categoria')
        print(f'Variável em branco\n{df[Var_linha]}\n')

    for col in Colunas:
        # Gerar Tabelas Ponderadas de frequência absoluta
        tabela = pd.crosstab(df[Var_linha], df[col], values=df[Var_Pond], aggfunc='sum')
        tabelas_pond_freq_abs.append(tabela)

        # Gerar Tabelas Ponderadas de frequência relativa 
        tabela = tabela.div(tabela.sum())
        tabela = tabela.fillna(0)
        tabelas_pond.append(tabela)
        print(f'Tabela Ponderada Freq Rel: \n{tabela}\n')

        # Gerar Tabelas Sem Ponderação
        tabela = pd.crosstab(df[Var_linha], df[col], dropna=False)
        tabela = tabela.fillna(0)
        print(f'Tabela Sem Ponderação: \n{tabela}\n')
        if len(tabela) == 0:
            tabela = pd.DataFrame(0, index=df[Var_linha][pd.notna(df[Var_linha])].unique(), 
                                columns=df[col][pd.notna(df[col])].unique())
            print(f'Tabela Sem Ponderação: \n{tabela}\n')
        tabelas_sem_pond.append(tabela)

        # Gerar Tabelas para valores agrupados
        if 'var_agrupada' in df.columns:
            tabela = pd.crosstab(df['var_agrupada'], df[col], values=df[Var_Pond], aggfunc='sum')
            tabela = tabela.div(tabela.sum())
            print(f'Tabela Valores Agrupados: \n{tabela}\n')
            aux_tabelas_pond.append(tabela)

            # Tabelas sem ponderação
            tabela = pd.crosstab(df['var_agrupada'], df[col], dropna=False)
            print(f'Tabela Valores Agrupados Sem Ponderação: \n{tabela}\n')
            aux_tabelas_sem_pond.append(tabela)

    tabela_geral = pd.concat(tabelas_pond, axis=1)
    print(f'\ntabela_geral (Tabela Ponderada de frequência relativa):\n{tabela_geral}')
    tabelas_pond_freq_abs = pd.concat(tabelas_pond_freq_abs, axis=1)
    tabelas_sem_pond = pd.concat(tabelas_sem_pond, axis=1)
    tabelas_sem_pond = tabelas_sem_pond[tabelas_sem_pond.index.notna()]
    print(f'\ntabelas_sem_pond (Tabela de frequência absoluta):\n{tabelas_sem_pond}')

    # Trazendo a coluna de valores gerais
    valores_gerais_pond = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
    print(f'\nValores GERAL:\n{valores_gerais_pond}')
    percentual_geral = valores_gerais_pond.div(valores_gerais_pond.sum()).sort_index()
    print(f'\PERCENTUAL GERAL:\n{percentual_geral}')

    if 'var_agrupada' in df.columns:
        aux_tabelas_pond = pd.concat(aux_tabelas_pond, axis=1)
        aux_tabelas_sem_pond = pd.concat(aux_tabelas_sem_pond, axis=1)

        # Percentual para os valores gerais da variável agrupada
        valores_gerais_aux = pd.pivot_table(df, values=Var_Pond, index='var_agrupada', aggfunc='sum')
        percentual_geral_aux = valores_gerais_aux.div(valores_gerais_aux.sum()).sort_index()
        tabela_geral = pd.concat([tabela_geral, aux_tabelas_pond], axis=0)
        percentual_geral = pd.concat([percentual_geral, percentual_geral_aux], axis=0)
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
    else:
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)

    # Multiplicando cada linha pelo seu índice+1
    if TipoTabela == 'NPS':
        # Gerar o NPS
        nps = tabela_geral.loc['Promotor'] - tabela_geral.loc['Detrator']
        tabela_geral.loc['NPS'] = nps
        # Gerar as médias
        media_tabela = tabelas_pond_freq_abs.copy()
        for i in range(len(tabelas_pond_freq_abs)):
            media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
        media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

        # Cálculo da média geral
        df_limpo = df.dropna(subset=[Var_linha])
        media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
        media_geral = pd.Series(media_geral)
        media_tabela = pd.concat([media_geral, media_tabela], axis=0)
        media_tabela.index = tabela_geral.columns
        tabela_geral.loc['Media'] = media_tabela

    elif (TipoTabela == 'SATISFACAO') | (TipoTabela == 'IPA_10') :
        media_tabela = tabelas_pond_freq_abs.copy()
        for i in range(len(tabelas_pond_freq_abs)):
            media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
        media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

        # Cálculo da média geral
        df_limpo = df.dropna(subset=[Var_linha])
        media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
        media_geral = pd.Series(media_geral)
        media_tabela = pd.concat([media_geral, media_tabela], axis=0)

        media_tabela.index = tabela_geral.columns
        tabela_geral.loc['Media'] = media_tabela

    else:
        tabela_geral

    tabela_geral = tabela_geral.fillna(0)
    print(f"\n#=== TABELA GERAL - VERIFICAR RESULTADO ===#\n{tabela_geral}")

    # Valores para Base Ponderada
    tabelas_pond_freq_abs = tabelas_pond_freq_abs.fillna(0)
    soma_colunas = tabelas_pond_freq_abs.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    # base_ponderada.index = tabela_geral.columns
    tabela_geral.loc['Base Ponderada'] = base_ponderada

    # Valores para Base Sem Ponderar
    tabelas_sem_pond = tabelas_sem_pond.fillna(0)
    soma_colunas = pd.Series(tabelas_sem_pond.sum())
    print(f'{soma_colunas}\n')
    valores_gerais = df[Var_linha].value_counts().sort_index()
    base_sem_ponderar = pd.Series(valores_gerais.sum())
    base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
    print(f'base_sem_ponderar.index:\n{base_sem_ponderar.index}\n')
    print(f'tabela_geral.columns:\n{tabela_geral.columns}\n')
    print(f'tabelas_sem_pond.columns:\n{tabelas_sem_pond.columns}\n')
    base_sem_ponderar.index = tabela_geral.columns
    tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

    tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
    print(f'TABELA GERAL:\n{tabela_geral.iloc[:, 0:10]}\n')

Tabela Ponderada Freq Rel: 
ONDA    Nov24  Mar25     Jul25     Nov25
NPS_IT                                  
0.0       0.0    0.0  0.000000  0.001681
2.0       0.0    0.0  0.016669  0.000000
3.0       0.0    0.0  0.010305  0.000000
4.0       0.0    0.0  0.020589  0.000000
5.0       0.0    0.0  0.027290  0.025551
6.0       0.0    0.0  0.022525  0.025580
7.0       0.0    0.0  0.080510  0.073899
8.0       0.0    0.0  0.132229  0.194741
9.0       0.0    0.0  0.156501  0.171727
10.0      0.0    0.0  0.533382  0.506820

Tabela Sem Ponderação: 
ONDA    Nov24  Mar25  Jul25  Nov25
NPS_IT                            
NaN      3609   3826   3753   3820
 0.0        0      0      0      2
 2.0        0      0      2      0
 3.0        0      0      2      0
 4.0        0      0      2      0
 5.0        0      0      5      7
 6.0        0      0      3      8
 7.0        0      0      9     17
 8.0        0      0     18     62
 9.0        0      0     21     40
 10.0       0      0     62    118


In [22]:
Cabecalho = Cabecalho.split(sep=',')
print(f'Cabeçalho:\n{Cabecalho}')
header_above = []
print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
for col in tabela_geral.columns:
    valor = col.split(sep=' - ')[0]
    print(f'valor: {valor}')
    header_above.append(valor)
print(f'\nheader_above:\n{header_above}')

col_series = []
for i, valor in enumerate(Cabecalho):
    # col_names = df[Colunas[i]][pd.notna(df[Colunas[i]])].unique()
    col_names = dict_ord_labels[Colunas[i]]
    print(f'\ncol_names:\n{col_names}')
    for col in col_names:
        col_series.append((Titulo, valor, col))

header = [(Titulo, '', 'GERAL')]
print(f'\ncol_series:\n{col_series}')
header = header + col_series
print(f'\nheader:\n{header}')
print(f'\ntamanho header:\t{len(header)}')
    
header = pd.MultiIndex.from_tuples(header)
print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
print(f'\ntamanho tabela_geral.columns:\t{len(tabela_geral.columns)}')
tabela_geral.columns = header
tabela_geral
print(f'\n\n #===== TABELA GERAL FINAL =====#\n{tabela_geral}\n')
print(f'#===================================================================================#\n')

Cabeçalho:
['Ondas', ' Empreendedores (PF+PJ)', ' Varejo (PF+PJ)', ' Grandes Contas (PF+PJ)']

tabela_geral.columns:
Index(['Geral', 'Nov24', 'Mar25', 'Jul25', 'Nov25', 'Emp - Nov24',
       'Emp - Mar25', 'Emp - Jul25', 'Emp - Nov25', 'Varejo - Nov24',
       'Varejo - Mar25', 'Varejo - Jul25', 'Varejo - Nov25', 'GC - Nov24',
       'GC - Mar25', 'GC - Jul25', 'GC - Nov25'],
      dtype='object')
valor: Geral
valor: Nov24
valor: Mar25
valor: Jul25
valor: Nov25
valor: Emp
valor: Emp
valor: Emp
valor: Emp
valor: Varejo
valor: Varejo
valor: Varejo
valor: Varejo
valor: GC
valor: GC
valor: GC
valor: GC

header_above:
['Geral', 'Nov24', 'Mar25', 'Jul25', 'Nov25', 'Emp', 'Emp', 'Emp', 'Emp', 'Varejo', 'Varejo', 'Varejo', 'Varejo', 'GC', 'GC', 'GC', 'GC']

col_names:
['Nov24', 'Mar25', 'Jul25', 'Nov25']

col_names:
['Emp - Nov24', 'Emp - Mar25', 'Emp - Jul25', 'Emp - Nov25']

col_names:
['Varejo - Nov24', 'Varejo - Mar25', 'Varejo - Jul25', 'Varejo - Nov25']

col_names:
['GC - Nov24', 'GC - M

In [23]:
display(tabela_geral)

C26. Satisfação Geral Final com a CIELO               \
                                                                 Ondas   
                                                    GERAL        Nov24   
1.0                                              0.012730     0.019834   
2.0                                              0.004364     0.004575   
3.0                                              0.004163     0.002420   
4.0                                              0.009253     0.008138   
5.0                                              0.036767     0.037714   
6.0                                              0.034143     0.033716   
7.0                                              0.092576     0.090856   
8.0                                              0.229251     0.211313   
9.0                                              0.213646     0.217966   
10.0                                             0.363107     0.373468   
BTB                                              0.021257     0.026829   
Neutro                                           0.172739     0.170424   
TTB                                              0.806004     0.802747   
Media                                            8.495574     8.482534   
Base Ponderada                                8015.040033  1950.416138   
Base Sem Ponderar                             8016.000000  3609.000000   

                                                          \
                                                           
                         Mar25        Jul25        Nov25   
1.0                   0.014242     0.004998     0.011990   
2.0                   0.006441     0.003009     0.003268   
3.0                   0.002705     0.005796     0.005836   
4.0                   0.009471     0.009644     0.009737   
5.0                   0.037691     0.038773     0.032643   
6.0                   0.040390     0.034927     0.026783   
7.0                   0.106441     0.086361     0.085462   
8.0                   0.237807     0.236607     0.230229   
9.0                   0.207790     0.226633     0.201994   
10.0                  0.337024     0.353254     0.392058   
BTB                   0.023388     0.013803     0.021094   
Neutro                0.193992     0.169704     0.154624   
TTB                   0.782620     0.816494     0.824281   
Media                 8.391798     8.540016     8.577486   
Base Ponderada     2129.677775  2025.002680  1909.943440   
Base Sem Ponderar  3826.000000  3877.000000  4074.000000   

                                                                     \
                   Empreendedores (PF+PJ)                             
                              Emp - Nov24  Emp - Mar25  Emp - Jul25   
1.0                              0.011801     0.020083     0.011017   
2.0                              0.000630     0.007353     0.003014   
3.0                              0.006184     0.002970     0.007690   
4.0                              0.017230     0.012072     0.006578   
5.0                              0.048216     0.054352     0.041015   
6.0                              0.025026     0.042411     0.032675   
7.0                              0.079481     0.089755     0.105471   
8.0                              0.226869     0.236754     0.224992   
9.0                              0.197951     0.186671     0.206010   
10.0                             0.386612     0.347580     0.361538   
BTB                              0.018615     0.030406     0.021721   
Neutro                           0.169953     0.198589     0.185740   
TTB                              0.811432     0.771005     0.792540   
Media                            8.510765     8.296363     8.475253   
Base Ponderada                 228.907150   393.293385   325.073205   
Base Sem Ponderar             1563.000000  1804.000000  1672.000000   

                                                                              \
                                Varejo (

In [30]:
tabela_geral.iloc[1, 0]
for tuplaMI in tabela_geral.columns:
    print(tuplaMI[1])


Ondas
Ondas
Ondas
Ondas
 Empreendedores (PF+PJ)
 Empreendedores (PF+PJ)
 Empreendedores (PF+PJ)
 Empreendedores (PF+PJ)
 Varejo (PF+PJ)
 Varejo (PF+PJ)
 Varejo (PF+PJ)
 Varejo (PF+PJ)
 Grandes Contas (PF+PJ)
 Grandes Contas (PF+PJ)
 Grandes Contas (PF+PJ)
 Grandes Contas (PF+PJ)


In [ ]:
lista_labels

,Coluna,Codigo,Label
1,ONDA,29,Mai23 (mensal)
2,ONDA,30,Jun23 (mensal)
3,ONDA,31,Jul23
4,ONDA,32,Ago23 (mensal)
5,ONDA,33,Set23 (mensal)
...,...,...,...
10244,VAR_G,4,Varejo - Nov25
10245,GC_G,1,GC - Nov24
10246,GC_G,2,GC - Mar25
10247,GC_G,3,GC - Jul25


### <font color="greenblue">Criar a função modular</font>

In [ ]:
class processar_tabela:
    """Processamento de dados Estatístico
    Será informado os parâmetros necessários para a criação da tabela processada e assim,
    iniciaremos cada método necessário dentro da classe 'processar_tabela'
    """
    def __init__(self, bd_dados: pd.DataFrame, lista_labels: pd.DataFrame, 
                 TipoTabela: str, Var_linha: str, Colunas: list, Cabecalho: str, NS_NR: str, Var_ID: str, Var_Pond: str, Titulo: str,
                 valores_BTB: str = '1, 2, 3', valores_TTB: str = '8, 9, 10', Valores_Agrup: str = 'Q8_1, Q8_2, Q8_3', Fecha_100: str = 'SIM'):
        self.bd_dados = bd_dados
        self.lista_labels = lista_labels
        self.TipoTabela = TipoTabela
        self.Var_linha = Var_linha
        self.Colunas = Colunas
        self.Cabecalho = Cabecalho
        self.NS_NR = NS_NR
        self.Var_ID = Var_ID
        self.Var_Pond = Var_Pond
        self.Titulo = Titulo
        self.valores_BTB = valores_BTB
        self.valores_TTB = valores_TTB
        self.Valores_Agrup = Valores_Agrup
        self.Fecha_100 = Fecha_100

        df = self.bd_dados.copy()
        dict_ord_labels = {}

        for col in Colunas:
            if col not in df.columns:
                raise ValueError(f"Coluna '{col}' não encontrada no DataFrame.")        
            else:
                df[col], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=col)
                dict_ord_labels[col] = ord_labels
        
        return df
    

    def tratamento_var_linha(self, df):
        # Etapas de ETL para as colunas principais (nível das linhas) a serem utilizadas
        # Transformação na variável para a linha da tabela
        if TipoTabela == 'SIMPLES':
            if NS_NR == 'NAO':
                df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
                df[Var_linha], ord_labels = ordenar_labels(df=self.bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            else:
                df[Var_linha], ord_labels = ordenar_labels(df=self.bd_dados, lista_labels=lista_labels, Variavel=Var_linha)

            return df
            
        elif (TipoTabela == 'NPS') | (TipoTabela == 'IPA_10'):
            if NS_NR == 'NAO':
                df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
                df[Var_linha] = df[Var_linha].replace('ns/nr', np.nan)
                df[Var_linha] = df[Var_linha].replace('90', np.nan)
                df[Var_linha] = df[Var_linha].replace('99', np.nan)
                df[Var_linha] = df[Var_linha].replace('999', np.nan)
                df[Var_linha] = df[Var_linha].replace('9999', np.nan)
                df[Var_linha] = df[Var_linha].replace(90, np.nan)
                df[Var_linha] = df[Var_linha].replace(99, np.nan)
                df[Var_linha] = df[Var_linha].replace(999, np.nan)
                df[Var_linha] = df[Var_linha].replace(9999, np.nan)
                df[Var_linha] = pd.to_numeric(df[Var_linha], errors='coerce', downcast='integer')
                df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

                if TipoTabela == 'NPS':
                    df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

                elif TipoTabela == 'IPA_10':
                    valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                    valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                    df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

            else:
                df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

                if TipoTabela == 'NPS':
                    df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

                elif TipoTabela == 'IPA_10':
                    valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                    valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                    df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

            return df

        elif TipoTabela == 'IPA_5':
            if NS_NR == 'NAO':
                df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
                df[Var_linha] = df[Var_linha].replace('90', np.nan)
                df[Var_linha] = df[Var_linha].replace('99', np.nan)
                df[Var_linha] = df[Var_linha].replace('999', np.nan)
                df[Var_linha] = df[Var_linha].replace(90, np.nan)
                df[Var_linha] = df[Var_linha].replace(99, np.nan)
                df[Var_linha] = df[Var_linha].replace(999, np.nan)
                valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
                df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
                df[Var_linha], ord_labels = ordenar_labels(df=self.bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            else:
                valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
                df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
                df[Var_linha], ord_labels = ordenar_labels(df=self.bd_dados, lista_labels=lista_labels, Variavel=Var_linha)

            return df

        elif TipoTabela == 'MULTIPLA':
            if NS_NR == 'NAO':
                df_NS_NR = df.copy()
                
                Valores_Agrup = Valores_Agrup.split(sep=', ')

                bd_motivo = pd.melt(df, 
                            id_vars=Colunas + [Var_Pond] + [Var_ID],
                            value_vars=Valores_Agrup, 
                            var_name='Valores', 
                            value_name=Var_linha)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('90', np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('99', np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('999', np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('9999', np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(90, np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(99, np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(999, np.nan)
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(9999, np.nan)
                bd_motivo = bd_motivo.dropna(subset=[Var_linha])
                print(f'\nbd_motivo em formato de código:\n{bd_motivo}')
                bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
                bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('NS/NR', np.nan)

                df_limpo = bd_motivo.dropna(subset=[Var_linha])
                print(f'bd_motivo finalizado:\n{df_limpo}')
                df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')
                        
                # Bancos para realizar o cálculo do Índice de Multiplicidade (incluir NS/NR)
                bd_motivo_NS_NR = pd.melt(df_NS_NR, 
                            id_vars=Colunas + [Var_Pond] + [Var_ID],
                            value_vars=Valores_Agrup, 
                            var_name='Valores', 
                            value_name=Var_linha)
                bd_motivo_NS_NR = bd_motivo_NS_NR.dropna(subset=[Var_linha])
                bd_motivo_NS_NR = ordenar_labels_multipla(df=bd_motivo_NS_NR, lista_labels=lista_labels, Variavel=Var_linha, 
                                                        Var_Valores_Agrup=Valores_Agrup[0])
                
                df_NS_NR_limpo = bd_motivo_NS_NR.dropna(subset=[Var_linha])
                print(f'bd_motivo_NS_NR finalizado:\n{df_limpo}')
                df_NS_NR_unico = df_NS_NR_limpo.drop_duplicates(subset=Var_ID, keep='first')  

                return df_unico, df_NS_NR_unico  
                
            else:
                Valores_Agrup = Valores_Agrup.split(sep=', ')
                bd_motivo = pd.melt(df, 
                            id_vars=Colunas + [Var_Pond] + [Var_ID],
                            value_vars=Valores_Agrup, 
                            var_name='Valores', 
                            value_name=Var_linha)
                bd_motivo = bd_motivo.dropna(subset=[Var_linha])
                bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
                
                df_limpo = bd_motivo.dropna(subset=[Var_linha])
                df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')    

                return bd_motivo, df_unico
    


In [ ]:
# FUNÇÃO PARA PROCESSAR UMA ÚNICA TABELA
def processar_tabela(bd_dados: pd.DataFrame, lista_labels: pd.DataFrame, 
                     TipoTabela: str, Var_linha: str, Colunas: list, Cabecalho: str, NS_NR: str, Var_ID: str, Var_Pond: str, Titulo: str,
                     valores_BTB: str = '1, 2, 3', valores_TTB: str = '8, 9, 10', Valores_Agrup: str = 'Q8_1, Q8_2, Q8_3', Fecha_100: str = 'SIM'):
    
    df = bd_dados.copy()
    dict_ord_labels = {}

    for col in Colunas:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada no DataFrame.")        
        else:
            df[col], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=col)
            dict_ord_labels[col] = ord_labels
            print(f"\nColuna ordenada (unique): {df[col].unique()}")

    # Etapas de ETL para as colunas principais (nível das linhas) a serem utilizadas
    # Transformação na variável para a linha da tabela
    if TipoTabela == 'SIMPLES':
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
        else:
            df[Var_linha], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
        
    elif (TipoTabela == 'NPS') | (TipoTabela == 'IPA_10'):
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha] = df[Var_linha].replace('ns/nr', np.nan)
            df[Var_linha] = df[Var_linha].replace('90', np.nan)
            df[Var_linha] = df[Var_linha].replace('99', np.nan)
            df[Var_linha] = df[Var_linha].replace('999', np.nan)
            df[Var_linha] = df[Var_linha].replace('9999', np.nan)
            df[Var_linha] = df[Var_linha].replace(90, np.nan)
            df[Var_linha] = df[Var_linha].replace(99, np.nan)
            df[Var_linha] = df[Var_linha].replace(999, np.nan)
            df[Var_linha] = df[Var_linha].replace(9999, np.nan)
            df[Var_linha] = pd.to_numeric(df[Var_linha], errors='coerce', downcast='integer')
            # df[Var_linha], ord_labels = ordenar_labels(df=df, lista_labels=lista_labels, Variavel=Var_linha)
            df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

            if TipoTabela == 'NPS':
                df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

            elif TipoTabela == 'IPA_10':
                valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

        else:
            # df[Var_linha] = df[Var_linha].replace('90', np.nan)
            # df[Var_linha] = df[Var_linha].replace('99', np.nan)
            # df[Var_linha] = df[Var_linha].replace('999', np.nan)
            # df[Var_linha] = df[Var_linha].replace('9999', np.nan)
            # df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

            if TipoTabela == 'NPS':
                df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

            elif TipoTabela == 'IPA_10':
                valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

    elif TipoTabela == 'IPA_5':
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha] = df[Var_linha].replace('90', np.nan)
            df[Var_linha] = df[Var_linha].replace('99', np.nan)
            df[Var_linha] = df[Var_linha].replace('999', np.nan)
            df[Var_linha] = df[Var_linha].replace(90, np.nan)
            df[Var_linha] = df[Var_linha].replace(99, np.nan)
            df[Var_linha] = df[Var_linha].replace(999, np.nan)
            valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
            df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
            df[Var_linha], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)
        else:
            valores_BTB = [int(v) for v in valores_BTB.split(sep=', ')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=', ')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
            df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
            df[Var_linha], ord_labels = ordenar_labels(df=bd_dados, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)

    elif TipoTabela == 'MULTIPLA':
        if NS_NR == 'NAO':
            df_NS_NR = df.copy()
            
            Valores_Agrup = Valores_Agrup.split(sep=', ')
            # Converte as colunas de motivos para string, preservando NaN
            # for c in Valores_Agrup:
            #     df[c] = df[c].astype("object").where(df[c].isna(), df[c].astype(str))
            #     df_NS_NR[c] = df_NS_NR[c].astype("object").where(df_NS_NR[c].isna(), df_NS_NR[c].astype(str))

            bd_motivo = pd.melt(df, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('90', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('99', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('999', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('9999', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(90, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(99, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(999, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(9999, np.nan)
            bd_motivo = bd_motivo.dropna(subset=[Var_linha])
            print(f'\nbd_motivo em formato de código:\n{bd_motivo}')
            bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('NS/NR', np.nan)
            # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)  

            df_limpo = bd_motivo.dropna(subset=[Var_linha])
            print(f'bd_motivo finalizado:\n{df_limpo}')
            df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')
                    
            # Bancos para realizar o cálculo do Índice de Multiplicidade (incluir NS/NR)
            bd_motivo_NS_NR = pd.melt(df_NS_NR, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo_NS_NR = bd_motivo_NS_NR.dropna(subset=[Var_linha])
            bd_motivo_NS_NR = ordenar_labels_multipla(df=bd_motivo_NS_NR, lista_labels=lista_labels, Variavel=Var_linha, 
                                                    Var_Valores_Agrup=Valores_Agrup[0])
            # bd_motivo_NS_NR[Var_linha] = pd.Categorical(bd_motivo_NS_NR[Var_linha], 
            #                                     categories=ordenar_valores(bd_motivo_NS_NR[Var_linha]), ordered=True)
            
            df_NS_NR_limpo = bd_motivo_NS_NR.dropna(subset=[Var_linha])
            print(f'bd_motivo_NS_NR finalizado:\n{df_limpo}')
            df_NS_NR_unico = df_NS_NR_limpo.drop_duplicates(subset=Var_ID, keep='first')    
            
        else:
            Valores_Agrup = Valores_Agrup.split(sep=', ')
            bd_motivo = pd.melt(df, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo = bd_motivo.dropna(subset=[Var_linha])
            bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
            # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], 
            #                                     categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)
            
            df_limpo = bd_motivo.dropna(subset=[Var_linha])
            df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')


    #======= Criação da Tabela Geral =======#
    tabela_geral = []
    tabelas_pond = []
    tabelas_pond_freq_abs = []
    tabelas_sem_pond = []
    aux_tabelas_pond = []
    aux_tabelas_sem_pond = []

    tbl_pond_freq_abs_respondentes = []
    tbl_sem_pond_respondentes = []

    tbl_pond_freq_abs_respostas_NS_NR = []
    tbl_pond_freq_abs_respondentes_NS_NR = []

    if TipoTabela == 'MULTIPLA':
        if NS_NR == 'NAO':
            banco = df_NS_NR_unico
            banco_pivotado = bd_motivo_NS_NR
        else:
            banco = df_unico
            banco_pivotado = bd_motivo

        i = 0    
        for col in Colunas:

            # Gerar Tabelas Ponderadas de Frequência Absoluta com o banco empilhado
            tabela = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            if Fecha_100 == 'SIM':
                tabela = tabela.div(tabela.sum())
                tabelas_pond.append(tabela)
            else:
                tabelas_pond.append(tabela)

            tabela = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respostas_NS_NR.append(tabela)

            tabela = pd.pivot_table(df_unico, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respondentes.append(tabela)

            tabela = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respondentes_NS_NR.append(tabela)

            tabela = pd.crosstab(df_unico[Var_linha], df_unico[col], dropna=False)
            tabela = tabela.fillna(0)
            if len(tabela) == 0:
                tabela = pd.DataFrame(0, index=df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique(), 
                                    columns=df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique())
                print(f'{df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique()}\n')
                print(f'{df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique()}\n')
            print(f'tabela:\n{tabela}')
            tbl_sem_pond_respondentes.append(tabela)
            i += 1

        tabela_geral = pd.concat(tabelas_pond, axis=1)
        tbl_pond_freq_abs_respondentes = pd.concat(tbl_pond_freq_abs_respondentes, axis=1)
        tbl_sem_pond_respondentes = pd.concat(tbl_sem_pond_respondentes, axis=1)
        tbl_pond_freq_abs_respostas_NS_NR = pd.concat(tbl_pond_freq_abs_respostas_NS_NR, axis=1)
        tbl_pond_freq_abs_respondentes_NS_NR = pd.concat(tbl_pond_freq_abs_respondentes_NS_NR, axis=1)
        
        if Fecha_100 != 'SIM':
            soma_base_ponderada = tbl_pond_freq_abs_respondentes.sum()
            tabela_geral = tabela_geral.div(soma_base_ponderada)
        
        print(f'{tabela_geral}\n')


        # Trazendo a coluna de valores gerais
        soma_colunas = tbl_pond_freq_abs_respondentes.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        valores_gerais = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, aggfunc='sum')
        if Fecha_100 == 'SIM':
            percentual_geral = valores_gerais.div(valores_gerais.sum()).sort_index()
            print(f'\npercentual_geral:\n{percentual_geral}')
            print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')
        else:
            percentual_geral = valores_gerais.div(base_ponderada[0]).sort_index()
            print(f'\npercentual_geral:\n{percentual_geral}')
            print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')

        # tabela_geral = tabela_geral.div(base_ponderada[1:])
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
        print(f'{tabela_geral}\n')

        tabela_geral_front = tabela_geral.multiply(100)
        tabela_geral_front = tabela_geral_front.applymap(lambda x: f"{x:.1f}%".replace(".", ","))

        # Valores para Base Ponderada
        soma_colunas = tbl_pond_freq_abs_respondentes.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        tabela_geral.loc['Base Ponderada'] = base_ponderada
        tabela_geral_front.loc['Base Ponderada'] = base_ponderada.round().astype(int)

        # Valores para Base Sem Ponderar
        valores_gerais_respondentes = df_unico[Var_ID].value_counts()
        soma_colunas = pd.Series(tbl_sem_pond_respondentes.sum())
        print(f'\n{soma_colunas}\n')
        base_sem_ponderar = pd.Series(valores_gerais_respondentes.sum())
        base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
        print(f'base sem ponderar:\n{base_sem_ponderar.index}\n')
        print(f'{len(base_sem_ponderar.index)}\n')
        print(f'tabela geral:\n{tabela_geral.columns}\n')
        print(f'{len(tabela_geral.columns)}\n')
        base_sem_ponderar.index = tabela_geral.columns
        tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar
        tabela_geral_front.loc['Base Sem Ponderar'] = base_sem_ponderar

        # Valores para Base Ponderada - Respostas
        soma_colunas_respostas = tbl_pond_freq_abs_respostas_NS_NR.sum()
        soma_colunas_respostas = pd.Series(soma_colunas_respostas)
        base_ponderada_respostas = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, aggfunc='sum')
        base_ponderada_respostas = pd.Series(base_ponderada_respostas.sum())
        base_ponderada_respostas = pd.concat([base_ponderada_respostas, soma_colunas_respostas])

        # Valores para Base Ponderada - Respondentes
        soma_colunas = tbl_pond_freq_abs_respondentes_NS_NR.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])

        # Índice de Multiplicidade (Total de respostas / Total de respondentes)
        indice_multiplicidade = base_ponderada_respostas / base_ponderada
        tabela_geral.loc['Índice de Multiplicidade'] = indice_multiplicidade
        tabela_geral_front.loc['Índice de Multiplicidade'] = indice_multiplicidade

        tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
        print(f'{tabela_geral}\n')
        tabela_geral_front.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)

    else:
        if df[Var_linha].isna().all():
            # df[Var_linha] = df[Var_linha].astype('float').fillna(0)
            df[Var_linha] = df[Var_linha].cat.add_categories(['Não possui categoria']).fillna('Não possui categoria')
            print(f'Variável em branco\n{df[Var_linha]}\n')

        for col in Colunas:
            # Gerar Tabelas Ponderadas de frequência absoluta
            tabela = pd.crosstab(df[Var_linha], df[col], values=df[Var_Pond], aggfunc='sum')
            tabelas_pond_freq_abs.append(tabela)

            # Gerar Tabelas Ponderadas de frequência relativa 
            tabela = tabela.div(tabela.sum())
            tabela = tabela.fillna(0)
            tabelas_pond.append(tabela)
            print(f'Tabela Ponderada Freq Rel: \n{tabela}\n')

            # Gerar Tabelas Sem Ponderação
            tabela = pd.crosstab(df[Var_linha], df[col], dropna=False)
            tabela = tabela.fillna(0)
            print(f'Tabela Sem Ponderação: \n{tabela}\n')
            if len(tabela) == 0:
                tabela = pd.DataFrame(0, index=df[Var_linha][pd.notna(df[Var_linha])].unique(), 
                                    columns=df[col][pd.notna(df[col])].unique())
                print(f'Tabela Sem Ponderação: \n{tabela}\n')
            tabelas_sem_pond.append(tabela)

            # Gerar Tabelas para valores agrupados
            if 'var_agrupada' in df.columns:
                tabela = pd.crosstab(df['var_agrupada'], df[col], values=df[Var_Pond], aggfunc='sum')
                tabela = tabela.div(tabela.sum())
                print(f'Tabela Valores Agrupados: \n{tabela}\n')
                aux_tabelas_pond.append(tabela)

                # Tabelas sem ponderação
                tabela = pd.crosstab(df['var_agrupada'], df[col], dropna=False)
                print(f'Tabela Valores Agrupados Sem Ponderação: \n{tabela}\n')
                aux_tabelas_sem_pond.append(tabela)

        tabela_geral = pd.concat(tabelas_pond, axis=1)
        tabelas_pond_freq_abs = pd.concat(tabelas_pond_freq_abs, axis=1)
        tabelas_sem_pond = pd.concat(tabelas_sem_pond, axis=1)

        # Trazendo a coluna de valores gerais
        valores_gerais_pond = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
        print(f'\nValores GERAL:\n{valores_gerais_pond}')
        percentual_geral = valores_gerais_pond.div(valores_gerais_pond.sum()).sort_index()
        print(f'\PERCENTUAL GERAL:\n{percentual_geral}')

        if 'var_agrupada' in df.columns:
            aux_tabelas_pond = pd.concat(aux_tabelas_pond, axis=1)
            aux_tabelas_sem_pond = pd.concat(aux_tabelas_sem_pond, axis=1)

            # Percentual para os valores gerais da variável agrupada
            valores_gerais_aux = pd.pivot_table(df, values=Var_Pond, index='var_agrupada', aggfunc='sum')
            percentual_geral_aux = valores_gerais_aux.div(valores_gerais_aux.sum()).sort_index()
            tabela_geral = pd.concat([tabela_geral, aux_tabelas_pond], axis=0)
            percentual_geral = pd.concat([percentual_geral, percentual_geral_aux], axis=0)
            tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)

            tabela_geral_front = tabela_geral.multiply(100)
            tabela_geral_front = tabela_geral_front.applymap(lambda x: f"{x:.1f}%".replace(".", ","))

        else:
            tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)

            tabela_geral_front = tabela_geral.multiply(100)
            tabela_geral_front = tabela_geral_front.applymap(lambda x: f"{x:.1f}%".replace(".", ","))

        # Multiplicando cada linha pelo seu índice+1
        if TipoTabela == 'NPS':
            # Gerar o NPS
            nps = tabela_geral.loc['Promotor'] - tabela_geral.loc['Detrator']
            tabela_geral.loc['NPS'] = nps
            # Gerar as médias
            media_tabela = tabelas_pond_freq_abs.copy()
            for i in range(len(tabelas_pond_freq_abs)):
                media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
            media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

            # Cálculo da média geral
            df_limpo = df.dropna(subset=[Var_linha])
            media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
            media_geral = pd.Series(media_geral)
            media_tabela = pd.concat([media_geral, media_tabela], axis=0)
            media_tabela.index = tabela_geral.columns

            tabela_geral_front = tabela_geral.multiply(100)
            tabela_geral_front = tabela_geral_front.applymap(lambda x: f"{x:.1f}%".replace(".", ","))
            tabela_geral_front.loc['Media'] = media_tabela

            tabela_geral.loc['Media'] = media_tabela

        elif (TipoTabela == 'SATISFACAO') | (TipoTabela == 'IPA_10') :
            media_tabela = tabelas_pond_freq_abs.copy()
            for i in range(len(tabelas_pond_freq_abs)):
                media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
            media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

            # Cálculo da média geral
            df_limpo = df.dropna(subset=[Var_linha])
            media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
            media_geral = pd.Series(media_geral)
            media_tabela = pd.concat([media_geral, media_tabela], axis=0)

            media_tabela.index = tabela_geral.columns

            tabela_geral_front = tabela_geral.multiply(100)
            tabela_geral_front = tabela_geral_front.applymap(lambda x: f"{x:.1f}%".replace(".", ","))
            tabela_geral_front.loc['Media'] = media_tabela

            tabela_geral.loc['Media'] = media_tabela

        else:
            tabela_geral

        tabela_geral = tabela_geral.fillna(0)

        # Valores para Base Ponderada
        tabelas_pond_freq_abs = tabelas_pond_freq_abs.fillna(0)
        soma_colunas = tabelas_pond_freq_abs.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        # base_ponderada.index = tabela_geral.columns
        tabela_geral.loc['Base Ponderada'] = base_ponderada
        tabela_geral_front.loc['Base Ponderada'] = base_ponderada.round().astype(int)

        # Valores para Base Sem Ponderar
        tabelas_sem_pond = tabelas_sem_pond.fillna(0)
        soma_colunas = pd.Series(tabelas_sem_pond.sum())
        print(f'{soma_colunas}\n')
        valores_gerais = df[Var_linha].value_counts().sort_index()
        base_sem_ponderar = pd.Series(valores_gerais.sum())
        base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
        print(f'{base_sem_ponderar.index}\n')
        print(f'{tabela_geral.columns}\n')
        base_sem_ponderar.index = tabela_geral.columns
        tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar
        tabela_geral_front.loc['Base Sem Ponderar'] = base_sem_ponderar

        tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
        tabela_geral_front.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
        print(f'TABELA GERAL:\n{tabela_geral.iloc[:, 0:10]}\n')


    Cabecalho = Cabecalho.split(sep=', ')
    print(f'Cabeçalho:\n{Cabecalho}')
    header_above = []
    print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
    for col in tabela_geral.columns:
        valor = col.split(sep=' - ')[0]
        print(f'valor: {valor}')
        header_above.append(valor)
    print(f'\nheader_above:\n{header_above}')

    col_series = []
    for i, valor in enumerate(Cabecalho):
        # col_names = df[Colunas[i]][pd.notna(df[Colunas[i]])].unique()
        col_names = dict_ord_labels[Colunas[i]]
        print(f'\ncol_names:\n{col_names}')
        for col in col_names:
            col_series.append((Titulo, valor, col))

    header = [(Titulo, '', 'GERAL')]
    print(f'\ncol_series:\n{col_series}')
    header = header + col_series
    print(f'\nheader:\n{header}')
    print(f'\ntamanho header:\t{len(header)}')
        
    header = pd.MultiIndex.from_tuples(header)
    print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
    print(f'\ntamanho tabela_geral.columns:\t{len(tabela_geral.columns)}')
    tabela_geral.columns = header
    tabela_geral_front.columns = header
    print(f"Tabela processada:\n{tabela_geral}")
    
    return tabela_geral, tabela_geral_front

In [ ]:
df = pd.DataFrame({
    "Label": ["Empresa 1", "Empresa 1", "Empresa 2", "Empresa 2"],
    "Codigo": [1, 2, 2, 2]
})
df

: 

In [25]:
import pandas as pd
import numpy as np
from collections import Counter
from utils import ordenar_labels, ordenar_labels_multipla, ordenar_valores, classificar_nps, funcao_agrupamento, classificar_satis
path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\Cielo"
data = pd.read_excel(path + "\EMP_Cielo Satisfacao_3onda_NOV25_2025.12.26.xlsx", sheet_name="BD_CODIGOS")
lista_labels = pd.read_excel(path + "\EMP_Cielo Satisfacao_3onda_NOV25_2025.12.26.xlsx", sheet_name="LISTA_LABELS", header=1)
lista_labels.columns = ['Coluna', 'Codigo', 'Label']
lista_labels["Coluna"] = lista_labels["Coluna"].ffill().str.strip()
lista_labels

,Coluna,Codigo,Label
0,ONDA,28.0,Abr23
1,ONDA,29.0,Mai23 (mensal)
2,ONDA,30.0,Jun23 (mensal)
3,ONDA,31.0,Jul23
4,ONDA,32.0,Ago23 (mensal)
...,...,...,...
10221,TEMP_AC,6.0,8 anos ou mais
10222,TEMP_AC,90.0,NS/NR
10223,SEXO,1.0,Masculino
10224,SEXO,2.0,Feminino


In [59]:
sintaxe = pd.read_excel(path + "\Sintaxe_Cielo_Satisfacao_Nov25.xlsx", sheet_name="Teste2")

In [ ]:
def verif_TipoTabela(TipoTabela):
        if TipoTabela not in ["SIMPLES", "IPA_5", "IPA_10", "NPS", "MULTIPLA"]:
            return 1
        return 0

def verif_bandeiras_cabecalho(Bandeiras, Cabecalho):
     Bandeiras = Bandeiras.split(", ")
     Cabecalho = Cabecalho.split(", ")
     qtd_bandeiras = len(Bandeiras)
     qtd_cabecalho = len(Cabecalho)
     if qtd_bandeiras != qtd_cabecalho:
          return 1
     return 0

def verif_bandeiras(df, Bandeiras):
     Bandeiras = Bandeiras.split(", ")
     for col in Bandeiras:
          if col not in df.columns:
            # raise ValueError(f"⚠️ Coluna '{col}' não encontrada no DataFrame.")
            return 1, col
     return 0, col
      
def verif_Var_linha(df, Var_linha, TipoTabela):
     if TipoTabela != "MULTIPLA":
          if Var_linha not in df.columns:
               return 1
     return 0

def verif_NS_NR(NS_NR):
        if NS_NR not in ["NAO", "SIM"]:
            return 1
        return 0

def verif_Fecha_100(TipoTabela, Fecha_100):
     if TipoTabela == "MULTIPLA":
          if Fecha_100 not in ["NAO", "SIM"]:
               return 1
     return 0

def verif_coluna_existe(df, coluna):
    if coluna not in df.columns:
        return 1
    return 0


class verificar_incosistencias_iniciais:
    def __init__(self, data, sintaxe, lista_labels):
        self.data = data
        self.sintaxe = sintaxe
        self.lista_labels = lista_labels
    
    def verificar_incosistencia(self):
        df = self.data.copy()
        for line in range(len(self.sintaxe)):
            i = 0
            TipoTabela = self.sintaxe.iloc[line, i]
            i+=1
            Bandeiras = self.sintaxe.iloc[line, i]
            i+=1
            Cabecalho = self.sintaxe.iloc[line, i]
            i+=1
            Var_linha = self.sintaxe.iloc[line, i]
            i+=1
            NS_NR = self.sintaxe.iloc[line, i]
            i+=1
            valores_BTB = self.sintaxe.iloc[line, i]
            i+=1
            valores_TTB = self.sintaxe.iloc[line, i]
            i+=1
            Valores_Agrup = self.sintaxe.iloc[line, i]
            i+=1
            Fecha_100 = self.sintaxe.iloc[line, i]
            i+=1
            Var_ID = self.sintaxe.iloc[line, i]
            i+=1
            Var_Pond = self.sintaxe.iloc[line, i]
            i+=1
            Titulo = self.sintaxe.iloc[line, i]

            res_TipoTabela = verif_TipoTabela(TipoTabela)
            if res_TipoTabela == 1:
                 return f"⚠️ Verificar incosistência: o **Tipo de Tabela** informado na linha {line+2} não corresponde com as opções válidas: [SIMPLES, IPA_5, IPA_10, NPS, MULTIPLA]"
            
            res_bandeiras_cabecalho = verif_bandeiras_cabecalho(Bandeiras, Cabecalho)
            if res_bandeiras_cabecalho == 1:
                 return f"⚠️ Verificar incosistência: na linha {line+2}, **o nº de Bandeiras não é compatível com o nº de inputs do Cabeçalho**."
            
            res_bandeira, col = verif_bandeiras(df, Bandeiras)
            if res_bandeira == 1:
                 return f"⚠️ Coluna **{col}** que se encontra na coluna **Bandeiras** não foi encontrada no Banco de dados."
            
            res_Var_linha = verif_Var_linha(df, Var_linha, TipoTabela)
            if res_Var_linha == 1:
                 return f"⚠️ Verificar incosistência: na linha {line+2}, a variável/coluna informada que representa o **nível da linha** da tabela não consta no Banco de dados."
            
            res_NS_NR = verif_NS_NR(NS_NR)
            if res_NS_NR == 1:
                 return f"⚠️ Verificar incosistência: o valor informado na linha {line+2} da coluna **Contabiliza_NS/NR** não corresponde com as opções válidas: [NAO, SIM]"
            
            res_Fecha_100 = verif_Fecha_100(TipoTabela, Fecha_100)
            if res_Fecha_100 == 1:
                 return(f"⚠️ Verificar incosistência: o valor informado na linha {line+2} da coluna **Fecha_100** não corresponde com as opções válidas: [NAO, SIM]")
            
            res_Var_ID = verif_coluna_existe(df, Var_ID)
            if res_Var_ID == 1:
                 return f"⚠️ Verificar incosistência: na linha {line+2}, a variável/coluna informada que representa o **Código de identificação da entrevista** não consta no Banco de dados."
            
            res_Var_Pond = verif_coluna_existe(df, Var_Pond)
            if res_Var_Pond == 1:
                 return f"⚠️ Verificar incosistência: na linha {line+2}, a variável/coluna informada que representa a **Ponderação** não consta no Banco de dados."
        return 0           


In [61]:
processamento_sintaxe = verificar_incosistencias_iniciais(data=data, sintaxe=sintaxe, lista_labels=lista_labels)
processamento_sintaxe.verificar_incosistencia()

'⚠️ Verificar incosistência: o valor informado na linha 2 da coluna **Contabiliza_NS/NR** não corresponde com as opções válidas: [NAO, SIM]'

In [1]:
def add_item(item, lst=[]):
    lst.append(item)
    return lst

add_item(1)
add_item(2)
print(add_item(3))

[1, 2, 3]


In [11]:
lista = [['SEG_ONDA', 'PESSOA_ONDA']] * 9
print(lista)

lista_final = []
for valor in lista:
    lista_final.append(valor[0] + ", " + valor[1])
    print(valor[0] + ", " + valor[1])

print(lista_final)

[['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA'], ['SEG_ONDA', 'PESSOA_ONDA']]
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
SEG_ONDA, PESSOA_ONDA
['SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA', 'SEG_ONDA, PESSOA_ONDA']


In [13]:
lista = [['ONDA', 'SEG_ONDA', 'PESSOA_ONDA']] * 9
lista_final = [", ".join(valor) for valor in lista]

for item in lista_final:
    print(item)

ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA
ONDA, SEG_ONDA, PESSOA_ONDA


In [1]:
import pandas as pd
df = pd.DataFrame({
    "ColA": [1, 1, 2],
    "ColB": ["Jan", "Jan", "Fev"]
})
mapping_de_para = dict(zip(df['ColA'], df['ColB']))
mapping_de_para

{1: 'Jan', 2: 'Fev'}